# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 dataset using the `mlcroissant` library. We will:
- Load Croissant metadata and records
- Identify available record sets and fields, referencing all entities by their `@id`
- Extract and analyze tabular data
- Apply basic exploratory data analysis (EDA) and visualize the dataset

### Dataset Source
The dataset is described by a Croissant schema, available at:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset's metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print metadata summary
print(f"Dataset: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Let's review all available record sets, their fields, and column-level details, referencing their `@id` fields.

In [ ]:
# List available RecordSets and fields, referencing by @id

print("Available RecordSets in this dataset:")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"\n  RecordSet @id: {rs.id}")
    print(f"    name: {rs.name}")
    print(f"    description: {rs.description}")
    print(f"    Fields and Columns:")
    for field in rs.fields:
        print(f"      Field @id: {field.id} | name: {field.name}")
        for col in field.columns:
            print(f"        Column @id: {col.id} | name: {col.name}")

## 3. Data Extraction
We'll extract the record sets into DataFrames. All references to record sets, fields, and columns are made via their `@id` values. 

Let's demonstrate by extracting all available tabular record sets:

In [ ]:
# Extract all record sets into DataFrames (referencing by @id)
dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Display the columns of the first record set as an example
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"Columns for record set {example_record_set_id}:\n{dataframes[example_record_set_id].columns.tolist()}")
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's process the data in a sample record set. We'll choose a suitable numeric field and demonstrate common EDA operations: filtering, normalization, and grouping. 

**Note**: All field/column references are made via their `@id`. Modify or extend this cell for your specific analysis.

In [ ]:
# Select a RecordSet and numeric Field/Column for analysis (replace these @id values as appropriate from your overview above)
record_set_id = None
numeric_field_id = None
group_field_id = None

if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    # Try to auto-detect numeric columns
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    other_candidates = [col for col in df.columns if col != numeric_field_id]
    if other_candidates:
        group_field_id = other_candidates[0]

if record_set_id and numeric_field_id:
    threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records in {record_set_id} with {numeric_field_id} > {threshold:.3f}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id} (mean of numeric fields):")
        display(grouped_df.head())
else:
    print("No numeric field detected for EDA. Please choose relevant field @ids from the overview section.")

## 5. Visualization
We'll plot a histogram of the selected numeric field and scatterplot if possible (referencing by `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id} (RecordSet: {record_set_id})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Scatterplot if there's a second numeric field
    if len(numeric_candidates) > 1:
        plt.figure(figsize=(6,6))
        sns.scatterplot(
            x=df[numeric_candidates[0]],
            y=df[numeric_candidates[1]]
        )
        plt.xlabel(numeric_candidates[0])
        plt.ylabel(numeric_candidates[1])
        plt.title(f"{numeric_candidates[0]} vs {numeric_candidates[1]}")
        plt.show()
    elif group_field_id:
        # Boxplot as alternative
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough numeric columns available for plotting. Please review the dataset structure.")

## 6. Conclusion
In this notebook, we've:
- Accessed FAIR^2 mass spectrometry dataset metadata and explored its structure using record set, field and column `@id`s
- Loaded the tabular data and demonstrated filtering, normalization, and grouping based on references to schema IDs
- Visualized the main numeric property distributions

This workflow provides a reproducible path to FAIR dataset processing using `mlcroissant`. Modify or extend field and record set `@id` variables as required for your research goals.